<a href="https://colab.research.google.com/github/cozen03/2026-Programming/blob/main/%EA%B2%8C%EC%9E%84%EC%9E%A5%EB%A5%B4%EC%B6%94%EC%B2%9C%EC%B1%97%EB%B4%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install -q streamlit==1.29.0
from google.colab import userdata
api_key = userdata.get('GPT_API')
with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={api_key}\n')


In [ ]:
%%writefile gpt_functions.py

def recommend_game_genre(vibe: str, play_time: str, player_count: str):

    v = vibe.replace(" ", "")
    t = play_time.replace(" ", "")
    p = player_count.replace(" ", "")

    # 1) 느낌(vibe) 기반 후보 장르 매칭
    if any(k in v for k in ["스트레스", "긴장", "빡센", "타격감", "손맛"]):
        candidates = ["액션", "슈팅(FPS)", "격투"]
        reason = "스트레스 해소에는 손맛과 타격감이 확실한 장르가 잘 맞아요."
    elif any(k in v for k in ["힐링", "느긋", "편안", "잔잔"]):
        candidates = ["시뮬레이션(농장/생활)", "퍼즐", "캐주얼"]
        reason = "편안하고 느긋한 느낌에는 부담 없이 즐길 수 있는 장르가 잘 맞아요."
    elif any(k in v for k in ["머리", "두뇌", "전략", "고민"]):
        candidates = ["전략(RTS/턴제)", "퍼즐", "로그라이크"]
        reason = "두뇌를 써야 하는 느낌에는 판단과 계획이 중요한 장르가 잘 맞아요."
    elif any(k in v for k in ["몰입", "스토리", "세계관", "감동"]):
        candidates = ["RPG", "어드벤처", "비주얼노벨"]
        reason = "몰입감 있는 스토리를 원한다면 서사 중심 장르가 잘 맞아요."
    elif any(k in v for k in ["무섭", "공포", "긴장감있는"]):
        candidates = ["공포(호러)", "생존"]
        reason = "긴장감 넘치는 분위기를 원한다면 공포/생존 장르가 잘 맞아요."
    else:
        candidates = ["어드벤처", "캐주얼", "퍼즐"]
        reason = "특별한 느낌이 지정되지 않아 무난하게 즐기기 좋은 장르로 골라봤어요."

    # 2) 플레이 시간(play_time)으로 후보 좁히기
    short_time = any(k in t for k in ["30분", "짧", "10분", "20분", "잠깐"])
    long_time = any(k in t for k in ["장시간", "길", "3시간", "몇시간", "오래"])

    if short_time:
        candidates = [c for c in candidates if c not in ["RPG", "전략(RTS/턴제)"]] or candidates
        note_time = "짧게 즐길 수 있는 라이트한 게임 위주로 골랐어요."
    elif long_time:
        note_time = "긴 호흡으로 즐길 수 있는 게임도 함께 고려했어요."
    else:
        note_time = "플레이 시간에 크게 구애받지 않는 장르들이에요."

    # 3) 인원수(player_count)로 최종 보정
    solo = any(k in p for k in ["혼자", "1명", "솔로"])
    multi = any(k in p for k in ["2명", "3명", "여러", "듀오", "팀", "멀티", "친구"])

    if multi:
        candidates.append("파티게임")
        note_people = "여럿이 함께 즐기기 좋도록 파티게임도 추천에 추가했어요."
    elif solo:
        note_people = "혼자서도 충분히 즐길 수 있는 장르들이에요."
    else:
        note_people = ""

    # 중복 제거, 상위 3개만 반환
    final_genres = list(dict.fromkeys(candidates))[:3]

    result = {
        "recommended_genres": final_genres,
        "reason": " ".join([reason, note_time, note_people]).strip(),
    }
    print(result)
    return str(result)


tools = [
    {
        "type": "function",
        "function": {
            "name": "recommend_game_genre",
            "description": "사용자가 원하는 게임의 느낌, 플레이 가능 시간, 함께 플레이할 인원수를 바탕으로 알맞은 게임 장르를 추천할 때 호출하는 함수입니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "vibe": {
                        "type": "string",
                        "description": "사용자가 원하는 게임의 느낌 (예: \"스트레스 풀고 싶어\", \"힐링되는 거\", \"머리 쓰는 거\", \"몰입감 있는 스토리\")",
                    },
                    "play_time": {
                        "type": "string",
                        "description": "플레이 가능한 시간 (예: \"30분\", \"짧게\", \"장시간\", \"3시간 이상\")",
                    },
                    "player_count": {
                        "type": "string",
                        "description": "함께 플레이할 인원수 (예: \"혼자\", \"1명\", \"2명\", \"3명 이상\")",
                    },
                },
                "required": ["vibe", "play_time", "player_count"],
            },
        }
    },
]


if __name__ == "__main__":
    print(recommend_game_genre(vibe="스트레스 풀고 싶어", play_time="30분", player_count="혼자"))


In [ ]:
from gpt_functions import recommend_game_genre, tools
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_ai_response(messages, tools=None):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
    )
    return response


messages = [
    {"role": "system", "content": (
        "너는 사용자 취향에 맞는 게임 '장르'를 추천해주는 게임 큐레이터야. "
        "사용자가 원하는 게임의 느낌(vibe), 플레이 가능 시간(play_time), 함께할 인원수(player_count) "
        "세 가지 정보가 모이면 반드시 recommend_game_genre 함수를 호출해서 장르를 추천해줘. "
        "정보가 부족하면 먼저 되물어보고, 절대 함수 호출 없이 임의로 장르를 추천하지 마. "
            "함수 실행 결과로 받은 장르만 답변에 사용하고, 함수 결과에 없는 장르는 절대 추가하거나 섞지 마."
    )},
]

while True:
    user_input = input("사용자\t: ")
    if user_input == "exit":
        break

    messages.append({"role": "user", "content": user_input})
    ai_response = get_ai_response(messages, tools=tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_call_id = tool_call.id
            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "recommend_game_genre":
                messages.append({
                    "role": "function",
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    "content": recommend_game_genre(
                        vibe=arguments["vibe"],
                        play_time=arguments["play_time"],
                        player_count=arguments["player_count"],
                    ),
                })
        messages.append({"role": "system", "content": "이제 주어진 결과를 바탕으로 답변할 차례다."})
        ai_response = get_ai_response(messages, tools=tools)
        ai_message = ai_response.choices[0].message

    messages.append(ai_message)
    print("AI\t: " + ai_message.content)


In [ ]:
%%writefile game_chatbot.py
from gpt_functions import recommend_game_genre, tools
from openai import OpenAI
import os
import json
import streamlit as st
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def get_ai_response(messages, tools=None):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
    )
    return response

st.title("🎮 게임 장르 추천 챗봇")

if "messages" not in st.session_state:
    st.session_state["messages"] = [
        {"role": "system", "content": (
            "너는 사용자 취향에 맞는 게임 '장르'를 추천해주는 게임 큐레이터야. "
            "사용자가 원하는 게임의 느낌(vibe), 플레이 가능 시간(play_time), 함께할 인원수(player_count) "
            "세 가지 정보가 모이면 반드시 recommend_game_genre 함수를 호출해서 장르를 추천해줘. "
            "정보가 부족하면 먼저 되물어보고, 절대 함수 호출 없이 임의로 장르를 추천하지 마. "
            "함수 실행 결과로 받은 장르만 답변에 사용하고, 함수 결과에 없는 장르는 절대 추가하거나 섞지 마."
        )},
        {"role": "assistant", "content": "안녕하세요! 오늘 어떤 느낌의 게임을 하고 싶으신가요? 😊"},
    ]

for msg in st.session_state.messages:
    if msg["role"] in ["assistant", "user"]:
        st.chat_message(msg["role"]).write(msg["content"])

if user_input := st.chat_input():
    st.session_state.messages.append({"role": "user", "content": user_input})
    st.chat_message("user").write(user_input)

    ai_response = get_ai_response(st.session_state.messages, tools=tools)
    ai_message = ai_response.choices[0].message
    print(ai_message)

    tool_calls = ai_message.tool_calls
    if tool_calls:
        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_call_id = tool_call.id
            arguments = json.loads(tool_call.function.arguments)

            if tool_name == "recommend_game_genre":
                st.session_state.messages.append({
                    "role": "function",
                    "tool_call_id": tool_call_id,
                    "name": tool_name,
                    "content": recommend_game_genre(
                        vibe=arguments["vibe"],
                        play_time=arguments["play_time"],
                        player_count=arguments["player_count"],
                    ),
                })
        st.session_state.messages.append({"role": "system", "content": "이제 주어진 결과를 바탕으로 답변할 차례다."})
        ai_response = get_ai_response(st.session_state.messages, tools=tools)
        ai_message = ai_response.choices[0].message

    st.session_state.messages.append({
        "role": "assistant",
        "content": ai_message.content
    })

    print("AI\t: " + ai_message.content)
    st.chat_message("assistant").write(ai_message.content)


In [ ]:
import urllib
print("LocalTunnel 접속용 Password (IP 주소):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))


In [ ]:
!streamlit run game_chatbot.py --server.enableCORS false --server.enableXsrfProtection false & npx -y localtunnel --port 8501
